# Pharmacy Claims Fraud Detection - EDA

Exploratory analysis of 50,000 synthetic pharmacy reimbursement claims to understand claim patterns and identify fraud signals.

In [ ]:
import os, sys
from pathlib import Path
# Ensure CWD is project root regardless of where Jupyter is launched from
nb_dir = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.')
project_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
# Simple fallback: if data/raw exists relative to parent, use parent
if not (Path('data/raw/claims.csv')).exists():
    candidate = Path('..') / 'data' / 'raw' / 'claims.csv'
    if candidate.exists():
        os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid')
PRIMARY = '#2C7BB6'
CHARTS = Path('outputs/charts')
CHARTS.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv('data/raw/claims.csv', parse_dates=['claim_date'])
df_scored = pd.read_csv('data/processed/scored_claims.csv', parse_dates=['claim_date'])
print(f'Raw: {len(df_raw):,} rows | Scored: {len(df_scored):,} rows')

## Chart 1 - Claim Amount Distribution

The distribution is right-skewed with a long tail of high-value outliers; claims above ~400 euros warrant closer scrutiny as potential upcoding cases.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(df_raw['claim_amount'], bins=60, color=PRIMARY, edgecolor='white', alpha=0.85)
mean_val = df_raw['claim_amount'].mean()
ax.axvline(mean_val, color='#D7191C', linewidth=2, linestyle='--', label=f'Mean E{mean_val:.0f}')
ax.set_title('Claim Amount Distribution', fontsize=16, fontweight='bold', pad=14)
ax.set_xlabel('Claim Amount (E)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(CHARTS / 'claim_amount_distribution.png', dpi=150)
plt.show()
print('Chart 1 saved.')

## Chart 2 - Fraud Rate by Provider Category

Hospitals and Specialists show the highest fraud rates, likely driven by higher baseline claim amounts making upcoding harder to detect without z-score normalisation.

In [ ]:
fraud_by_cat = (df_raw.groupby('provider_category')['is_fraud_ground_truth']
                .mean().sort_values(ascending=False).reset_index())
fraud_by_cat.columns = ['provider_category', 'fraud_rate']
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(fraud_by_cat['provider_category'], fraud_by_cat['fraud_rate'] * 100,
       color=PRIMARY, edgecolor='white')
ax.set_title('Fraud Rate by Provider Category', fontsize=16, fontweight='bold', pad=14)
ax.set_xlabel('Provider Category', fontsize=12)
ax.set_ylabel('Fraud Rate (%)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.savefig(CHARTS / 'fraud_by_category.png', dpi=150)
plt.show()
print('Chart 2 saved.')

## Chart 3 - Monthly Claim Volume

Claim volumes are relatively stable across 2022-2023 with no strong seasonality, suggesting fraud patterns are not concentrated in specific months.

In [ ]:
df_raw['month'] = df_raw['claim_date'].dt.to_period('M')
monthly = df_raw.groupby('month').size().reset_index(name='claim_count')
monthly['month_str'] = monthly['month'].astype(str)
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(monthly['month_str'], monthly['claim_count'], color=PRIMARY, linewidth=2.5, marker='o', markersize=5)
ax.fill_between(monthly['month_str'], monthly['claim_count'], alpha=0.15, color=PRIMARY)
ax.set_title('Monthly Claim Volume 2022-2023', fontsize=16, fontweight='bold', pad=14)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Number of Claims', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(CHARTS / 'monthly_claim_volume.png', dpi=150)
plt.show()
print('Chart 3 saved.')

## Chart 4 - Top 15 Highest Risk Providers

A small cluster of providers consistently scores in the top risk tier; these warrant priority investigation as they show multiple concurrent fraud signals.

In [ ]:
top_providers = (df_scored.groupby('provider_id')['risk_score']
                 .mean().sort_values(ascending=False).head(15).reset_index())
top_providers.columns = ['provider_id', 'avg_risk_score']
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_providers['provider_id'][::-1], top_providers['avg_risk_score'][::-1],
        color=PRIMARY, edgecolor='white')
ax.set_title('Top 15 Highest Risk Providers', fontsize=16, fontweight='bold', pad=14)
ax.set_xlabel('Average Risk Score (0-100)', fontsize=12)
ax.set_ylabel('Provider ID', fontsize=12)
plt.tight_layout()
plt.savefig(CHARTS / 'top_risky_providers.png', dpi=150)
plt.show()
print('Chart 4 saved.')

## Chart 5 - Risk Score Distribution with Flagging Threshold

The bimodal distribution reveals a clear separation between routine claims and high-risk outliers; the top-8% threshold captures the right tail while keeping manual review scope manageable.

In [ ]:
threshold = df_scored['risk_score'].quantile(0.92)
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(df_scored['risk_score'], bins=60, color=PRIMARY, edgecolor='white', alpha=0.85)
ax.axvline(threshold, color='#D7191C', linewidth=2.5, linestyle='--',
           label=f'Top 8% Threshold ({threshold:.1f})')
ax.set_title('Risk Score Distribution with Flagging Threshold', fontsize=16, fontweight='bold', pad=14)
ax.set_xlabel('Risk Score (0-100)', fontsize=12)
ax.set_ylabel('Number of Claims', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(CHARTS / 'risk_score_distribution.png', dpi=150)
plt.show()
print('Chart 5 saved.')
print('All charts saved to outputs/charts/')